# 🌤️ Laboratorio – Creación de un Dataset de Meteorología

En este laboratorio crearemos un dataset fusionando datos de dos fuentes abiertas:

- **Open-Meteo API**: datos meteorológicos históricos (sin clave)
- **AEMET** (si es posible descargar datos públicos)

Exploraremos los datos, los fusionaremos por fecha y hora, y haremos un pequeño análisis exploratorio.

## 1. Instalación de librerías (si es necesario)

In [ ]:
# Puedes ejecutar esto si trabajas en Colab o entorno limpio
# !pip install pandas requests matplotlib seaborn

## 2. Descarga de datos desde Open-Meteo

In [ ]:
import requests
import pandas as pd

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 37.39,      # Sevilla
    "longitude": -5.99,
    "start_date": "2023-09-01",
    "end_date": "2023-09-10",
    "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m",
    "timezone": "Europe/Madrid"
}

r = requests.get(url, params=params)
data = r.json()
df_openmeteo = pd.DataFrame(data['hourly'])
df_openmeteo['time'] = pd.to_datetime(df_openmeteo['time'])
df_openmeteo.head()

## 3. Carga de datos desde AEMET (si se dispone de CSV)

In [ ]:
# Este CSV puede descargarse manualmente de AEMET (datos abiertos) si no hay acceso API
# Aquí simulamos su carga desde un CSV local de ejemplo
# Reemplaza con tu archivo real

try:
    df_aemet = pd.read_csv('aemet_sample.csv', parse_dates=['fecha'])
    df_aemet.rename(columns={'fecha': 'time'}, inplace=True)
    display(df_aemet.head())
except:
    print("⚠️ Archivo de AEMET no encontrado. Puedes subir uno si lo tienes.")

## 4. Fusión por fecha y hora (si ambas fuentes están disponibles)

In [ ]:
if 'df_aemet' in locals():
    df_merged = pd.merge(df_openmeteo, df_aemet, on='time', how='inner')
    display(df_merged.head())
else:
    df_merged = df_openmeteo.copy()
    print("Solo Open-Meteo disponible. Seguimos con este dataset.")

## 5. Visualización y EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,4))
sns.lineplot(x='time', y='temperature_2m', data=df_merged)
plt.title('Temperatura a lo largo del tiempo')
plt.xticks(rotation=45)
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,4))
sns.lineplot(x='time', y='relative_humidity_2m', data=df_merged)
plt.title('Humedad relativa a lo largo del tiempo')
plt.xticks(rotation=45)
plt.grid()
plt.tight_layout()
plt.show()

## 6. Guardado del dataset final

In [ ]:
df_merged.to_csv('dataset_meteo.csv', index=False)
df_merged.to_parquet('dataset_meteo.parquet', index=False)
print('✅ Dataset guardado como CSV y Parquet.')